# Module 4: Multilayer Perceptron
Forward propagation through a 3-5-5-2-2 network.

| Layer | Neurons | Activation |
|---|---|---|
| Input | 3 | - |
| Hidden 1 | 5 | Sigmoid |
| Hidden 2 | 5 | Sigmoid |
| Hidden 3 | 2 | Sigmoid |
| Output (Layer 4) | 2 | ReLU |

The class lives in `multilayer_perceptron.py`; this notebook imports it.

## 1. Import and create the network

In [1]:
import numpy as np
from multilayer_perceptron import MultilayerPerceptron

mlp = MultilayerPerceptron()
for name in ('W1', 'W2', 'W3', 'W4'):
    b = 'b' + name[1]
    print(f"{name}: {mlp.network[name].shape}   {b}: {mlp.network[b].shape}")

W1: (3, 5)   b1: (5,)
W2: (5, 5)   b2: (5,)
W3: (5, 2)   b3: (2,)
W4: (2, 2)   b4: (2,)


## 2. Weights and biases (hand-chosen, fixed)

In [2]:
for k, v in mlp.network.items():
    print(k)
    print(v)
    print()

W1
[[ 0.1  0.2  0.3  0.4  0.5]
 [ 0.6  0.7  0.8  0.9  1. ]
 [-0.1 -0.2 -0.3 -0.4 -0.5]]

b1
[0.1 0.2 0.3 0.4 0.5]

W2
[[ 0.1 -0.2  0.3 -0.4  0.5]
 [ 0.2  0.3 -0.4  0.5 -0.6]
 [-0.3  0.4  0.5 -0.6  0.7]
 [ 0.4 -0.5  0.6  0.7 -0.8]
 [-0.5  0.6 -0.7  0.8  0.9]]

b2
[ 0.1 -0.1  0.1 -0.1  0.1]

W3
[[ 0.3 -0.7]
 [ 0.5  0.2]
 [-0.4  0.6]
 [ 0.8 -0.1]
 [-0.2  0.4]]

b3
[0.1 0.2]

W4
[[ 0.9  2. ]
 [ 0.4 -2. ]]

b4
[ 0.05 -0.09]



## 3. Forward propagation for one input
Each layer computes `a = f(a_prev @ W + b)`: sigmoid for layers 1-3, ReLU for layer 4.

In [3]:
x = np.array([1.0, 0.5, -1.5])
y, layers = mlp.forward(x, return_all=True)

names = ['input', 'layer 1 (sigmoid)', 'layer 2 (sigmoid)', 'layer 3 (sigmoid)', 'layer 4 (ReLU)']
for n, a in zip(names, layers):
    print(f"{n:18s}: {np.round(a, 4)}")
print()
print('Network output:', y)

input             : [ 1.   0.5 -1.5]
layer 1 (sigmoid) : [0.657  0.7408 0.81   0.8641 0.9047]
layer 2 (sigmoid) : [0.491  0.6048 0.5722 0.7006 0.6624]
layer 3 (sigmoid) : [0.6789 0.6261]
layer 4 (ReLU)    : [0.9115 0.0157]

Network output: [0.91146677 0.01567357]


## 4. Step-by-step check of the output layer
The pre-activation `z4` can be negative; ReLU clips those entries to 0.

In [4]:
net = mlp.network
a3 = layers[3]
z4 = a3 @ net['W4'] + net['b4']
print('z4 (before ReLU):', np.round(z4, 4))
print('relu(z4)        :', np.round(mlp.relu(z4), 4))
print('matches forward():', np.allclose(mlp.relu(z4), y))

z4 (before ReLU): [0.9115 0.0157]
relu(z4)        : [0.9115 0.0157]
matches forward(): True


## 5. Several example inputs
Note the third input: its second output pre-activation is negative, so ReLU returns 0.

In [5]:
inputs = np.array([
    [1.0, 0.5, -1.5],
    [0.0, 0.0, 0.0],
    [2.0, -1.0, 3.0],
    [-1.0, 2.0, 0.5],
])

print(f"{'input':>22s}   {'z4 (pre-ReLU)':>22s}   {'output':>18s}")
for x in inputs:
    y, layers = mlp.forward(x, return_all=True)
    z4 = layers[3] @ net['W4'] + net['b4']
    print(f"{str(x):>22s}   {str(np.round(z4, 4)):>22s}   {str(np.round(y, 4)):>18s}")

                 input            z4 (pre-ReLU)               output
      [ 1.   0.5 -1.5]          [0.9115 0.0157]      [0.9115 0.0157]
            [0. 0. 0.]          [0.8967 0.004 ]      [0.8967 0.004 ]
         [ 2. -1.  3.]        [ 0.877  -0.0136]        [0.877 0.   ]
      [-1.   2.   0.5]          [0.908  0.0082]      [0.908  0.0082]


## 6. Batch input and verification
`forward` also accepts a matrix of inputs (one row each). We check it against one-at-a-time results and against an independent calculation written without the class.

In [6]:
batch_out = mlp.forward(inputs)
print('batch output shape:', batch_out.shape)
print(np.round(batch_out, 4))

single_out = np.vstack([mlp.forward(x) for x in inputs])
print('batch == one-at-a-time:', np.allclose(batch_out, single_out))

def reference(x):
    sig = lambda z: 1 / (1 + np.exp(-z))
    a = x
    for i in (1, 2, 3):
        a = sig(a @ net[f'W{i}'] + net[f'b{i}'])
    return np.maximum(0, a @ net['W4'] + net['b4'])

ref_out = np.vstack([reference(x) for x in inputs])
print('matches independent calculation:', np.allclose(batch_out, ref_out))
print('all outputs >= 0 (ReLU):', bool((batch_out >= 0).all()))

batch output shape: (4, 2)
[[0.9115 0.0157]
 [0.8967 0.004 ]
 [0.877  0.    ]
 [0.908  0.0082]]
batch == one-at-a-time: True
matches independent calculation: True
all outputs >= 0 (ReLU): True
